# Reach Detection Accuracy Summary Table

## What question this answers

What fraction of reach start/end boundaries land exactly on GT, within 1 frame, within 5,
within 10, or beyond 10 -- and how many reaches are FP (phantom) or FN (miss)?

## Column definitions (NON-overlapping buckets)

| Column | Definition |
|--------|------------|
| delta=0 (%) | Matched with exactly 0 frame error |
| |delta|=1 (%) | Matched with exactly 1 frame error |
| 2<=|delta|<=5 (%) | Matched with 2-5 frames error |
| 6<=|delta|<=10 (%) | Matched with 6-10 frames error |
| FP | False positives (algo reaches with no GT match) |
| FN | False negatives (GT reaches with no algo match) |
| median |delta| | Median absolute error (matched only) |
| mean signed delta | Mean signed error (matched only) |

## What improvement looks like

- delta=0 fraction increases.
- Tail buckets shrink.
- FP and FN drop.
- median |delta| decreases.

In [ ]:
# === PARAMS ===
from pathlib import Path

SNAPSHOT_DIR = Path(r"Y:\2_Connectome\Behavior\MouseReach_Pipeline\Improvement_Snapshots\reach_detection\reach_v7.1.0_visibility_direction_reversal")
FIGSIZE = (16, 5)
DPI = 300
SAVE = False  # Flip to True to write PNG + legend

In [ ]:
# === LOAD ===
import json
import pandas as pd
import numpy as np

matches_path = SNAPSHOT_DIR / "metrics" / "reach_matches.csv"
scalars_path = SNAPSHOT_DIR / "metrics" / "scalars.json"

df_all = pd.read_csv(matches_path)
with open(scalars_path, "r") as f:
    scalars = json.load(f)

df_matched = df_all[df_all["status"] == "matched"].copy()
df_matched["start_delta"] = df_matched["start_delta"].astype(int)
df_matched["end_delta"] = df_matched["end_delta"].astype(int)

n_fp = scalars["total"]["n_fp"]
n_fn = scalars["total"]["n_fn"]
n_matched = scalars["total"]["n_matched"]
n_gt = scalars["total"]["n_gt"]
n_algo = scalars["total"]["n_algo"]
n_videos = scalars["n_videos"]
n_perfect = scalars["n_perfect_videos"]

print(f"Loaded {len(df_all)} rows, {len(df_matched)} matched")

In [ ]:
# === COMPUTE ===

rows = []
for delta_col, label in [("start_delta", "Start delta"), ("end_delta", "End delta")]:
    deltas = df_matched[delta_col].values
    abs_d = np.abs(deltas)

    n_d0 = int((abs_d == 0).sum())
    n_d1 = int((abs_d == 1).sum())
    n_d2_5 = int(((abs_d >= 2) & (abs_d <= 5)).sum())
    n_d6_10 = int(((abs_d >= 6) & (abs_d <= 10)).sum())

    # Sanity: buckets must sum to n_matched
    n_rest = n_matched - n_d0 - n_d1 - n_d2_5 - n_d6_10

    def pct(n):
        return round(100 * n / n_matched, 1) if n_matched > 0 else 0.0

    median_abs = float(np.median(abs_d)) if len(abs_d) > 0 else float("nan")
    mean_signed = float(np.mean(deltas)) if len(deltas) > 0 else float("nan")

    rows.append({
        "Delta Type": label,
        "delta=0 (%)": pct(n_d0),
        "|delta|=1 (%)": pct(n_d1),
        "2-5 (%)": pct(n_d2_5),
        "6-10 (%)": pct(n_d6_10),
        "FP": n_fp,
        "FN": n_fn,
        "med|d|": median_abs,
        "mean d": round(mean_signed, 2),
    })

table_df = pd.DataFrame(rows)

# Overall counts header
print(f"n_videos={n_videos}  n_gt={n_gt}  n_algo={n_algo}  n_matched={n_matched}  n_perfect={n_perfect}")
print()
print(table_df.to_string(index=False))

In [ ]:
# === RENDER ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=FIGSIZE, dpi=DPI,
                                      gridspec_kw={"height_ratios": [1, 2.5]})

# --- Top section: overall counts ---
ax_top.axis("off")
header_color = "#E8EAF6"
header_data = [
    ["n_videos", "n_gt_total", "n_algo_total", "n_matched", "n_perfect_videos"],
    [str(n_videos), str(n_gt), str(n_algo), str(n_matched), str(n_perfect)],
]
header_table = ax_top.table(
    cellText=[header_data[1]],
    colLabels=header_data[0],
    colColours=[header_color] * 5,
    loc="center",
    cellLoc="center",
)
header_table.auto_set_font_size(False)
header_table.set_fontsize(10)
header_table.scale(1, 1.6)
for (row_idx, col_idx), cell in header_table.get_celld().items():
    if row_idx == 0:
        cell.set_text_props(fontweight="bold", fontsize=9)
    cell.set_edgecolor("#CCCCCC")

# --- Bottom section: delta table ---
ax_bot.axis("off")

col_labels = list(table_df.columns)
cell_text = []
cell_colors = []

good_color = "#C8E6C9"
neutral_color = "#FFFFFF"

good_high_cols = {"delta=0 (%)"}
good_low_cols = {"2-5 (%)", "6-10 (%)"}

for _, row in table_df.iterrows():
    row_text = []
    row_colors = []
    for col in col_labels:
        val = row[col]
        if isinstance(val, float):
            if col in ("med|d|", "mean d"):
                row_text.append(f"{val:.1f}" if not (val != val) else "--")
            else:
                row_text.append(f"{val:.1f}")
        else:
            row_text.append(str(val))

        if col in good_high_cols:
            frac = min(float(val) / 100.0, 1.0) if isinstance(val, (int, float)) else 0
            r = int(255 - frac * (255 - 200))
            g = int(255 - frac * (255 - 230))
            b = int(255 - frac * (255 - 201))
            row_colors.append(f"#{r:02X}{g:02X}{b:02X}")
        elif col in good_low_cols:
            frac = min(float(val) / 20.0, 1.0) if isinstance(val, (int, float)) else 0
            if frac < 0.01:
                row_colors.append(good_color)
            else:
                r = int(200 + frac * (255 - 200))
                g = int(230 - frac * (230 - 205))
                b = int(201 - frac * (201 - 210))
                row_colors.append(f"#{r:02X}{g:02X}{b:02X}")
        else:
            row_colors.append(neutral_color)

    cell_text.append(row_text)
    cell_colors.append(row_colors)

table = ax_bot.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellColours=cell_colors,
    colColours=[header_color] * len(col_labels),
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.6)

for (row_idx, col_idx), cell in table.get_celld().items():
    if row_idx == 0:
        cell.set_text_props(fontweight="bold", fontsize=8)
    cell.set_edgecolor("#CCCCCC")

fig.suptitle("Reach detection accuracy summary", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === SAVE ===
if SAVE:
    fig_dir = SNAPSHOT_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    png_path = fig_dir / "summary_table.png"
    fig.savefig(str(png_path), dpi=DPI, bbox_inches="tight", pad_inches=0.15,
                facecolor="white")
    print(f"Saved: {png_path}")

    legend_md = f"""# Reach Detection Accuracy Summary Table

## What question this answers

What fraction of reach boundaries land exactly on GT, within 1 frame, within 5,
within 10, or beyond 10 -- and how many reaches are FP or FN?

## Layout

- **Top section**: overall counts (n_videos, n_gt_total, n_algo_total, n_matched, n_perfect_videos)
- **Bottom section**: two rows (Start delta, End delta) with non-overlapping accuracy buckets

## Column definitions (NON-overlapping buckets)

| Column | Definition |
|--------|------------|
| delta=0 (%) | Matched with exactly 0 frame error |
| |delta|=1 (%) | Matched with exactly 1 frame error |
| 2-5 (%) | Matched with 2-5 frames error |
| 6-10 (%) | Matched with 6-10 frames error |
| FP | False positives (algo reaches with no GT match) |
| FN | False negatives (GT reaches with no algo match) |
| med|d| | Median absolute error (matched only) |
| mean d | Mean signed error (matched only) |

## Rendering params

- SNAPSHOT_DIR: `{SNAPSHOT_DIR}`
- FIGSIZE: {FIGSIZE}
- DPI: {DPI}
"""
    legend_path = fig_dir / "summary_table_legend.md"
    legend_path.write_text(legend_md, encoding="utf-8")
    print(f"Saved: {legend_path}")
else:
    print("SAVE=False -- set SAVE=True to write PNG + legend")